# Ciência de Dados em Python — Guia geral de estudo (versão 0.5 — ampliada)

**Objetivo:** aprender os fundamentos desde o início e consultar explicações, comparações e exemplos durante exercícios. O conteúdo segue os cinco temas da disciplina, mas **não depende de questões particulares**. Os PDFs da Estácio orientam a cobertura dos Temas 3 e 4; as explicações incluem complementos conceituais quando necessário.

**Como usar:** leia o tema; pesquise palavras-chave com Ctrl+F; execute primeiro a célula de importações; altere somente os valores indicados nos exemplos. O índice alfabético está ao final.

**Roteiro:** 1. Fundamentos → 2. CNN → 3. Visão computacional → 4. PLN → 5. Reconhecimento de voz → Índice.

## Preparação — importações

**Execute a próxima célula uma vez** ao abrir o notebook. NumPy serve para matrizes; math para funções matemáticas; Counter para contagem de palavras. Se NumPy não estiver instalado, execute em outra célula: `%pip install numpy`. Os exemplos não exigem TensorFlow.

In [ ]:
import numpy as np
import math
from collections import Counter
try:
    import speech_recognition as sr  # opcional: %pip install SpeechRecognition
except ImportError:
    sr = None
print('Bibliotecas básicas importadas; SpeechRecognition:', 'disponível' if sr else 'não instalado')

## 1. Introdução — fundamentos

**IA** é o campo amplo de sistemas capazes de realizar tarefas associadas à inteligência. **Machine learning** aprende padrões a partir de dados. **Deep learning** é uma família de técnicas de ML que usa redes neurais profundas.

**Supervisionado:** aprende com entradas e respostas corretas (rótulos), por exemplo imagem → gato. **Não supervisionado:** identifica padrões sem rótulos, como agrupamentos. **Por reforço:** aprende por recompensas.

**Treino** é o conjunto usado para atualizar os pesos; **validação** é usado para acompanhar a qualidade e escolher hiperparâmetros, sem atualizar os pesos; **teste** mede desempenho final em dados não vistos. **Overfitting:** aprende detalhes demais do treino e piora em dados novos. Dropout, regularização, data augmentation e early stopping podem ajudar.

**Parâmetros** são aprendidos (pesos, vieses). **Hiperparâmetros** são escolhas de configuração (learning rate, momentum, número de épocas, tamanho do lote).

[Google — fundamentos de machine learning](https://developers.google.com/machine-learning/crash-course) · [Google — overfitting](https://developers.google.com/machine-learning/crash-course/overfitting/overview)

### 1.1 Como formular um problema de aprendizado de máquina

**A pergunta vem antes do algoritmo.** Para prever preço de um imóvel, a saída é um **número contínuo**: regressão. Para decidir se um e-mail é spam, a saída é uma **classe**: classificação. Para descobrir grupos de clientes sem rótulos, usa-se **agrupamento** (*clustering*). Para reduzir muitas variáveis a poucas representações, usa-se **redução de dimensionalidade**.

**Features/atributos** são as informações de entrada; **target/rótulo** é o resultado que se quer prever. **Amostra** é um exemplo do conjunto de dados. **Inferência** é utilizar o modelo já treinado para produzir uma previsão. **Generalização** é funcionar bem em dados que não participaram do treino. **Memorização não é generalização**.

**Viés e variância:** *underfitting* ocorre quando o modelo é simples demais para capturar o padrão (erro alto até no treino); *overfitting* ocorre quando se ajusta excessivamente aos dados de treino (erro baixo no treino, maior em validação). A complexidade ideal depende da quantidade, diversidade e qualidade dos dados.

**Vazamento de dados (*data leakage*):** informação do teste ou do futuro entra indevidamente no treinamento. Exemplo: calcular média de normalização usando treino e teste juntos antes de separar os conjuntos. Isso infla a avaliação. Em dados temporais, a divisão deve respeitar a ordem cronológica quando o objetivo é prever o futuro.

**Classificação binária × multiclasse × multirrótulo:** binária = duas classes; multiclasse = uma dentre várias classes mutuamente exclusivas; multirrótulo = várias classes simultâneas. Essa distinção determina a ativação e a função de perda adequadas.

### 1.2 Do conjunto de dados ao modelo implantado

1. **Definir objetivo e métrica:** prever classe? valor? localização? Acurácia pode ser inadequada em dados desbalanceados.
2. **Coletar e inspecionar:** verificar distribuição, dados faltantes, rótulos incorretos, duplicatas e representatividade.
3. **Separar treino/validação/teste:** treino aprende pesos; validação escolhe configurações; teste é reservado para estimativa final. *Cross-validation* é alternativa útil em bases menores.
4. **Preparar sem vazamento:** ajustar transformações no treino e aplicá-las à validação/teste. Aumento de dados, quando aplicável, normalmente fica no treino.
5. **Escolher baseline:** começar com método simples para ter referência. Uma rede profunda não é automaticamente melhor.
6. **Treinar e acompanhar:** registrar perdas e métricas por época, procurar underfitting/overfitting.
7. **Avaliar e implantar:** examinar erros por subgrupo, custo computacional, privacidade, viés e monitoramento após implantação.

**Métricas não são intercambiáveis:** MSE/MAE avaliam regressão; precisão, recall e F1 avaliam classificação; *loss* é o objetivo otimizado, que pode ou não coincidir com a métrica de negócio.

[Google — classificação e regressão](https://developers.google.com/machine-learning/crash-course) · [scikit-learn — validação cruzada](https://scikit-learn.org/stable/modules/cross_validation.html)

### 1.3 Exemplo executável: regressão linear sem biblioteca de ML

**O que faz:** ajusta uma reta $\hat y=wx+b$ para prever um valor numérico a partir de outro. O método dos mínimos quadrados escolhe $w$ e $b$ para reduzir a soma dos erros quadráticos. **Não é classificação** e não representa uma rede profunda; serve como referência para entender parâmetros aprendidos e previsões.

**Como usar:** altere os vetores $x$ e $y$ com observações correspondentes. O código mostra coeficientes e erro médio quadrático. A biblioteca NumPy já foi importada na primeira célula.

In [ ]:
x = np.array([1, 2, 3, 4, 5], dtype=float)  # ALTERE: atributo
y = np.array([2, 4, 5, 8, 10], dtype=float)  # ALTERE: valor esperado
assert x.shape == y.shape and len(x) >= 2
w, b = np.polyfit(x, y, deg=1)
previsoes = w * x + b
print(f"Modelo: y = {w:.3f} * x + {b:.3f}")
print("Previsões:", np.round(previsoes, 3))
print("MSE:", np.mean((y - previsoes) ** 2))

## 2. Redes neurais convolucionais (CNNs) — fundamentos explicados

### Pesos sinápticos: o que são?

Um neurônio artificial recebe entradas numéricas, multiplica cada entrada por um **peso sináptico** e soma os resultados com um **viés** (*bias*). O peso indica a influência de uma entrada; o nome é uma analogia com conexões entre neurônios biológicos. Exemplo: entradas 2 e 3, pesos 0,5 e −1, viés 1 → soma = 2×0,5 + 3×(−1) + 1 = −1. A função de ativação transforma essa soma.

**Os pesos são atualizados durante o treinamento? SIM.** O algoritmo calcula gradientes da função de erro e o otimizador modifica pesos e vieses para melhorar previsões. Em geral, na fase de teste/inferência, os pesos ficam fixos. **Não confunda:** pesos são parâmetros aprendidos; learning rate e momentum são hiperparâmetros.

### Treinamento: sequência de operações

1. **Forward pass:** a rede recebe os dados e produz previsões.
2. **Loss (função de erro):** compara previsões aos rótulos corretos.
3. **Backpropagation:** calcula como a loss muda em relação aos pesos (gradientes).
4. **Otimizador:** atualiza os pesos usando esses gradientes.
5. **Validação:** calcula previsões e loss em dados separados, **sem atualização dos pesos**.

**Learning rate (taxa de aprendizado)** controla o tamanho das atualizações de pesos. Pode ser fixada ou modificada ao longo do treinamento por um *scheduler*. Em geral, não é um parâmetro aprendido automaticamente como os pesos (embora existam métodos de otimização que adaptem taxas efetivas).\n\n**Momentum** incorpora a direção de atualizações anteriores, suaviza oscilações e **pode acelerar convergência**, mas não garante isso em todos os casos. Portanto, a frase 'momentum acelera a convergência' não é necessariamente falsa; em questão de resposta única, 'pesos são atualizados' descreve mais diretamente o treinamento.

**Fine-tuning** é continuar treinando um modelo pré-treinado, ajustando alguns ou todos os pesos.

### Função de erro (loss): quando é aplicada?

A loss **mede a diferença entre a previsão e a resposta correta**. É calculada no **treino** (para orientar atualização dos pesos) e na **validação** (para avaliar generalização, sem atualizar pesos). Também pode ser medida no teste como avaliação final. Logo, **não é exclusiva da fase de testes**. Exemplos: entropia cruzada para classificação, erro quadrático médio para regressão. **Loss não é ReLU, lambda ou normalização.**

[Google — redes neurais](https://developers.google.com/machine-learning/crash-course/neural-networks) · [PyTorch — otimização](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) · [Keras — losses](https://keras.io/api/losses/)

### Funções de ativação — explicação individual

Uma **função de ativação** transforma o valor produzido por neurônios ou camadas. Em camadas internas, muitas introduzem não linearidade, permitindo aprender padrões complexos. São usadas durante **treinamento e predição**; durante o treino influenciam gradientes, mas **não são o otimizador que atualiza pesos**.

**ReLU:** $f(x)=\\max(0,x)$. Valor negativo vira 0; positivo permanece. Comum em camadas internas de CNNs. Exemplo: [−2, 0, 3] → [0, 0, 3]. Não mede erro.

**Sigmoid:** $f(x)=1/(1+e^{-x})$. Produz valor entre 0 e 1; usada com frequência na saída de classificação binária e multirrótulo, quando cada classe é independente.

**Tanh:** produz valores entre −1 e 1, centrados em zero; comum em componentes de redes recorrentes.

**Softmax:** transforma um **vetor de escores brutos (logits)** em probabilidades não negativas cuja **soma é 1**. Fórmula: $softmax(z_i)=e^{z_i}/\\sum_j e^{z_j}$. Exemplo: logits [2, 1, 0] → aproximadamente [0,665; 0,245; 0,090]. É aplicada principalmente à **saída de classificação multiclasse exclusiva** (uma classe por imagem). Durante o treino, softmax pode fazer parte do cálculo da entropia cruzada; na predição, ajuda a interpretar as saídas. **Softmax não modifica diretamente pesos, não é loss e não é normalização dos dados de entrada.** A maior probabilidade indica a classe escolhida, não certeza absoluta.

**Resumo:** ReLU = zera negativos; sigmoid = valores 0–1 independentes; softmax = distribuição entre classes; loss = mede erro; otimizador = atualiza pesos.

[Google — funções de ativação](https://developers.google.com/machine-learning/crash-course/neural-networks/activation-functions) · [PyTorch — Softmax](https://docs.pytorch.org/docs/stable/generated/torch.nn.Softmax.html) · [Vídeo — redes neurais explicadas visualmente (3Blue1Brown)](https://www.youtube.com/watch?v=aircAruvnKk)

### Convolução e estrutura de uma CNN

Uma **CNN** aplica filtros (*kernels*) sobre regiões locais da imagem. **Convolução** combina dois conjuntos de números — entrada e filtro — para produzir um mapa de características (*feature map*). Em cada posição, multiplica elementos correspondentes e soma. Os valores dos filtros são **pesos aprendidos** durante o treinamento.

**Stride** é o passo de deslocamento; **padding** acrescenta bordas; **pooling/aglomeração** reduz a resolução espacial, por exemplo **max pooling** conserva o maior valor de cada região. **ReLU** frequentemente segue uma convolução. Arquitetura típica: imagem → blocos de convolução + ativação + pooling → classificador → previsão. Nem toda convolução precisa ser seguida imediatamente de pooling.

**Dimensão da saída** (sem dilatação): $O=\\lfloor(I+2P-K)/S\\rfloor+1$. I = tamanho da entrada; P = padding de cada lado; K = kernel; S = stride.

[Stanford — convolução e CNNs](https://cs231n.github.io/convolutional-networks/) · [TensorFlow — CNN passo a passo](https://www.tensorflow.org/tutorials/images/cnn)

### Exemplo de código A — calcular valores da convolução

**O que resolve:** dada uma matriz e um filtro, calcula os números produzidos quando o filtro percorre a matriz. Exemplo: na primeira posição, entrada [[1,2],[4,5]] e filtro [[1,0],[0,−1]] → 1×1+2×0+4×0+5×(−1)=−4. **Altere** `matriz` e `kernel` conforme o exercício. `stride` controla o deslocamento; `padding` acrescenta bordas. O padrão é a **correlação cruzada usada em CNNs**; `inverter_kernel=True` faz a convolução matemática com filtro invertido.

In [ ]:

def aplicar_filtro(matriz, kernel, stride=1, padding=0, inverter_kernel=False):
    x = np.asarray(matriz, dtype=float)
    k = np.asarray(kernel, dtype=float)
    if inverter_kernel:  # convolução matemática verdadeira
        k = np.flip(k)
    x = np.pad(x, padding)
    h, w = k.shape
    saida = [[np.sum(x[i:i+h, j:j+w] * k)
              for j in range(0, x.shape[1]-w+1, stride)]
             for i in range(0, x.shape[0]-h+1, stride)]
    return np.array(saida)

matriz = [[1,2,3],[4,5,6],[7,8,9]]  # ALTERE
kernel = [[1,0],[0,-1]]               # ALTERE
print(aplicar_filtro(matriz, kernel, stride=1, padding=0))
print('Convolução com kernel invertido:', aplicar_filtro(matriz, kernel, inverter_kernel=True))

### Exemplo de código B — calcular APENAS o tamanho da saída

**Este código não aplica o filtro nem calcula valores da matriz.** Ele determina quantas posições existirão em um eixo da saída. Exemplo: entrada 32, kernel 3, padding 1, stride 1 → saída 32. **Altere** I, K, P e S; para altura e largura diferentes, faça duas contas.

In [ ]:
I, K, P, S = 32, 3, 1, 1  # ALTERE para cada dimensão
O = math.floor((I + 2*P - K)/S) + 1
print('Tamanho da saída:', O)

### Camadas Dense (*fully connected*) — como a próxima camada interpreta a saída?

Uma **camada totalmente conectada (Dense)** possui neurônios que recebem **todas as ativações da camada anterior**. Se a entrada tem 4 valores e a Dense tem 3 neurônios, cada neurônio combina os quatro valores usando pesos próprios e um viés; a saída tem **3 números**. O cálculo é $y=f(Wx+b)$, onde $W$ é uma matriz de pesos aprendidos, $b$ são vieses e $f$ é a ativação.

**No meio da arquitetura**, a Dense não precisa gerar classes ou probabilidades: produz uma **representação intermediária** (*features aprendidas*). A camada seguinte simplesmente recebe esse vetor e aprende, durante o treinamento, combinações úteis. Exemplo: 10 entradas → Dense(8, ReLU) → Dense(4, ReLU) → Dense(2, softmax). As camadas transformam vetores de tamanhos 10 → 8 → 4 → 2; os quatro números intermediários não precisam corresponder a quatro conceitos humanos nomeáveis.

**No fim da arquitetura**, uma Dense pode produzir logits para classificação, um único valor para regressão ou outra representação. **A operação matemática é a mesma**; o papel depende da posição e da tarefa.

**Parâmetros de uma Dense:** com $d$ entradas e $h$ neurônios, há $d\times h$ pesos e $h$ vieses, total $d h+h$. Exemplo: 10 entradas → 8 neurônios = 88 parâmetros. **Camada Dense não é sinônimo de camada de saída.**

**Por que usar ativações?** Uma sequência de Dense sem ativações não lineares entre elas pode ser reduzida a uma única transformação afim. ReLU, tanh etc. permitem representar relações mais complexas. **Flatten** transforma mapa de características em vetor sem aprender pesos; **Reshape** reorganiza dimensões, também sem aprender pesos. Uma Conv2D após Dense exige um formato espacial adequado e uma representação aprendida para isso.

**O próximo código** calcula manualmente uma camada Dense com ReLU, sem framework adicional. Altere o vetor de entrada, os pesos e o viés, mantendo dimensões compatíveis.

In [ ]:
entrada_dense = np.array([0.8, 0.2, 0.9])  # ALTERE: 3 entradas
pesos_dense = np.array([[0.5, -0.3, 0.8], [-0.2, 0.6, 0.1]])  # 2 neurônios x 3 entradas
vieses_dense = np.array([0.1, -0.1])  # 1 viés por neurônio
assert pesos_dense.shape[1] == entrada_dense.size
assert pesos_dense.shape[0] == vieses_dense.size
logits_dense = pesos_dense @ entrada_dense + vieses_dense
saida_dense = np.maximum(0, logits_dense)  # ReLU
print("Antes da ativação:", logits_dense)
print("Saída para a próxima camada:", saida_dense)
print("Total de parâmetros:", pesos_dense.size + vieses_dense.size)

### 2.1 Treinamento de CNNs: escolhas e consequências

**Época:** passagem pelo conjunto de treino; **batch/lote:** subconjunto usado para calcular um gradiente; **iteração/step:** atualização dos parâmetros a partir de um lote. Com 1.000 exemplos e batch de 100, há aproximadamente 10 steps por época.

**Gradiente:** direção de maior aumento local da loss; descida do gradiente move os pesos na direção oposta. **SGD:** usa gradientes de lotes; **momentum:** acumula tendência de atualizações passadas; **Adam:** adapta a escala das atualizações usando estimativas de momentos. Learning rate muito alto pode causar instabilidade; muito baixo pode tornar treino lento. **Learning rate pode variar ao longo do treinamento** com *schedulers*; não é um parâmetro que necessariamente permanece fixo.

**Regularização:** L2 penaliza pesos grandes; dropout desativa aleatoriamente unidades durante treino; data augmentation altera exemplos preservando rótulos; early stopping interrompe treino quando validação deixa de melhorar. Dropout é desligado ou tratado diferentemente na inferência. **Batch normalization** normaliza ativações intermediárias usando estatísticas do lote durante treino e estatísticas acumuladas na inferência; não é igual a normalizar pixels de entrada.

**Transfer learning:** aproveitar pesos pré-treinados. **Extração de características:** congelar backbone e treinar uma nova cabeça. **Fine-tuning:** descongelar parte do backbone e continuar treino, normalmente com learning rate menor. **Congelar camada** significa não atualizar seus parâmetros, não removê-la do modelo.

**Funções de perda:** entropia cruzada binária para classes independentes; entropia cruzada categórica para uma classe entre várias; MSE para regressão. A loss é calculada no treino e pode ser acompanhada na validação, mas **somente a loss de treino orienta diretamente as atualizações normais de pesos**.

[PyTorch — treinamento e otimização](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) · [Keras — callbacks e early stopping](https://keras.io/api/callbacks/early_stopping/)

### VGGNet e GoogLeNet — diferenças decisivas

**VGGNet (VGG-16/VGG-19):** sua marca é usar **várias convoluções pequenas 3×3 empilhadas**. Elas são organizadas em **blocos**: duas ou três convoluções com ReLU e **depois** uma camada de **max pooling**. Isso se repete, aumentando canais e reduzindo resolução. **VGGNet original usa camadas completamente conectadas? SIM.** Na VGG-16 clássica há **13 camadas convolucionais + 3 totalmente conectadas = 16 camadas com pesos**. A última camada totalmente conectada produz os escores de classificação. Não é correto afirmar que toda convolução é imediatamente seguida por pooling: o pooling vem após grupos de convoluções.

**GoogLeNet (Inception v1):** é caracterizada pelos **módulos Inception** ('módulos de iniciação' em algumas traduções). **Dentro de cada módulo**, a mesma entrada percorre **ramos paralelos** com convoluções de diferentes tamanhos (como 1×1, 3×3 e 5×5) e pooling; as saídas são concatenadas. **Isso NÃO significa que todas as convoluções da rede sejam paralelas**: os módulos se sucedem e também existem camadas sequenciais. A rede original **tem dropout**, mas **dropout não é a característica que a define**. Ela usa *global average pooling* para reduzir parâmetros no classificador.

**Comparação para provas:** VGG = blocos de 3×3 + pooling + camadas fully connected; GoogLeNet = módulos Inception com ramos paralelos **internos**; ResNet = conexões residuais; LeNet = CNN histórica para dígitos; AlexNet = marco do ImageNet 2012.

**Armadilha:** 'há convoluções paralelas nos módulos Inception' é verdadeiro; 'todas as camadas convolucionais da GoogLeNet são paralelas' é falso.

[Artigo VGG original](https://arxiv.org/abs/1409.1556) · [Artigo GoogLeNet original](https://arxiv.org/abs/1409.4842) · [Keras — VGG](https://keras.io/api/applications/vgg/)

## 3. IA aplicada à visão computacional

### 3.1 O que é e onde se aplica

**Visão computacional** permite a um programa interpretar imagens e vídeos: identificar objetos, reconhecer padrões, localizar elementos e compreender cenas. **Processamento de imagens** transforma sinais visuais (redimensionar, filtrar, realçar, detectar bordas); **aprendizado de máquina** aprende relações a partir de exemplos. Os dois podem trabalhar juntos, mas não são sinônimos.

**Aplicações:** análise de exames médicos, monitoramento industrial, veículos autônomos, inspeção de produtos, segurança e análise de estoque no varejo. **Desafios:** iluminação variável, objetos parcialmente ocultos (*oclusão*), ruído, distorção, privacidade e generalização a imagens novas.

### 3.2 Como o computador representa uma imagem

Uma imagem digital pode ser representada por uma **matriz de pixels**. Em tons de cinza, cada posição registra intensidade; em RGB, há três canais (vermelho, verde e azul). **Extração de características** procura padrões como bordas, texturas, formas e cores. Filtros convolucionais extraem padrões locais; camadas posteriores podem combiná-los em características mais abstratas.

**Representação de objetos:** classificação atribui categoria à imagem; detecção identifica **onde** estão objetos (caixas delimitadoras); reconhecimento associa identidade ou categoria; **modelagem 3D** reconstrói geometria tridimensional a partir de imagens 2D. Uma cena também pode ser representada por objetos, posições e relações espaciais.

### 3.3 Segmentação: o que cada tipo entrega

**Segmentação de imagem** separa regiões. **Semântica:** atribui uma classe a **cada pixel**, por exemplo estrada, pedestre ou veículo; duas pessoas distintas recebem a mesma classe. **De instâncias:** além da classe, identifica **cada objeto individualmente** (pessoa A e pessoa B). **Panóptica:** combina regiões semânticas com identidades de objetos. Segmentação é útil em diagnóstico médico, agricultura, robótica e direção autônoma.

**Distinção importante:** classificação = um rótulo para a imagem; detecção = caixas; segmentação = rótulos por pixel; segmentação de instâncias = pixels + indivíduos.

[OpenCV — tutoriais de imagens](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html) · [TensorFlow — segmentação de imagens](https://www.tensorflow.org/tutorials/images/segmentation)

### 3.3.1 Imagens, pixels, canais e pré-processamento

Uma imagem RGB com altura H e largura W é um tensor H × W × 3. Os três canais representam intensidades de vermelho, verde e azul; em imagens de 8 bits por canal, cada intensidade normalmente varia de 0 a 255. Imagens em escala de cinza possuem um canal. **Resolução espacial** (quantidade de pixels) não é o mesmo que **profundidade de cor** (quantidade de valores representáveis).

**Redimensionamento** adapta dimensões; **normalização** pode dividir intensidades por 255; **padronização** subtrai a média e divide pelo desvio-padrão. **BGR** é uma ordem de canais comum no OpenCV e não deve ser confundida com RGB. **Interpolação bilinear** suaviza valores; **nearest-neighbor** é preferível para redimensionar máscaras de classes, pois não inventa rótulos intermediários.

**Filtros tradicionais:** Gaussiano suaviza e reduz ruído; Sobel detecta mudanças locais de intensidade (bordas). **Detecção de bordas não é detecção de objetos**: bordas podem pertencer a texturas ou ao fundo.

**Extração de características:** filtros iniciais podem responder a bordas e texturas; camadas posteriores combinam padrões em representações mais abstratas. Uma característica aprendida não precisa corresponder a um conceito humano nomeável.

[OpenCV — processamento de imagens](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html)

### 3.3.2 Localização, detecção, segmentação e métricas espaciais

**Classificação** produz um rótulo por imagem; **localização** indica a posição de um objeto; **detecção** produz caixas e classes de vários objetos; **segmentação semântica** rotula cada pixel por categoria; **segmentação de instâncias** distingue objetos individuais da mesma categoria; **segmentação panóptica** combina informação semântica e instâncias.

**Bounding box** pode ser descrita por (xmin, ymin, xmax, ymax) ou por (centro x, centro y, largura, altura). Não misture coordenadas em pixels e normalizadas entre 0 e 1.

**IoU (Intersection over Union)** é área da interseção dividida pela área da união entre duas caixas ou máscaras. IoU = 1 indica coincidência perfeita; IoU = 0 indica nenhuma sobreposição. **NMS** elimina caixas redundantes muito sobrepostas; **mAP** resume qualidade de detecção combinando precisão e critérios de localização. **Dice** mede sobreposição entre máscaras: 2 × interseção / (área A + área B).

**Arquiteturas comuns:** YOLO e Faster R-CNN para detecção; U-Net para segmentação semântica; Mask R-CNN para segmentação de instâncias. A escolha depende de velocidade, precisão, memória e características dos dados.

[TensorFlow — segmentação](https://www.tensorflow.org/tutorials/images/segmentation) · [U-Net — artigo](https://arxiv.org/abs/1505.04597)

### 3.4 Arquiteturas de visão computacional

**CNN:** aprende filtros locais e forma uma hierarquia de características. **Camada convolucional:** extrai padrões; **ativação:** introduz não linearidade; **pooling:** reduz resolução; **camada totalmente conectada:** combina características para classificação/regressão em arquiteturas clássicas. CNNs podem realizar classificação, detecção e segmentação.

**RNN:** rede recorrente que mantém estado ao processar sequências; embora associada a texto e áudio, pode analisar **sequências de quadros de vídeo** ou eventos visuais no tempo. **GAN:** possui **gerador** (cria imagens sintéticas) e **discriminador** (tenta distinguir imagens reais de geradas); ambos competem durante treinamento. Usos: síntese de imagens, aumento de dados e restauração. **Não confundir:** GAN gera dados; CNN é uma família de arquitetura frequentemente usada para extrair características; RNN modela sequências.

**Computação em nuvem:** permite armazenar dados e executar treinamento/inferência com recursos sob demanda; vantagens incluem **escalabilidade e flexibilidade**. É uma infraestrutura de execução, não uma arquitetura de rede neural. O desenvolvimento pode ocorrer localmente ou em plataformas de nuvem.

### 3.5 Ciclo de desenvolvimento de modelos

1. **Coleta e rotulagem:** imagens e respostas esperadas.
2. **Pré-processamento:** redimensionar, normalizar pixels (por exemplo dividir valores de 0–255 por 255), remover ruídos e selecionar regiões de interesse.
3. **Aumento de dados (*data augmentation*):** rotações, espelhamentos e outras transformações adequadas, normalmente **somente no treino**, para ampliar diversidade sem coletar novas imagens.
4. **Construção:** definir camadas, função de perda e otimizador.
5. **Treinamento supervisionado:** atualizar pesos a partir de imagens rotuladas e da perda.
6. **Validação e ajuste:** medir generalização, ajustar hiperparâmetros e arquitetura.
7. **Teste final e implantação:** avaliar em dados separados e usar em aplicações reais.

**Não confundir:** normalizar pixels é uma transformação de entrada; a função de erro compara previsão e rótulo; o otimizador atualiza pesos. O pré-processamento não se limita à normalização.

[TensorFlow — classificação de imagens](https://www.tensorflow.org/tutorials/images/classification) · [Google — conjuntos de dados](https://developers.google.com/machine-learning/crash-course/datasets)

### 3.6 Avaliação de classificadores — matriz de confusão

A **matriz de confusão** cruza classes verdadeiras com classes previstas e permite analisar erros por tipo. Para classificação binária:

| Real / Previsto | Positivo | Negativo |
|---|---:|---:|
| Positivo | VP (acerto positivo) | FN (positivo perdido) |
| Negativo | FP (falso alarme) | VN (acerto negativo) |

**Acurácia:** $(VP+VN)/(VP+VN+FP+FN)$ — proporção de acertos totais. **Precisão (*precision*):** $VP/(VP+FP)$ — entre previsões positivas, quantas estão certas? **Recall/sensibilidade:** $VP/(VP+FN)$ — entre positivos reais, quantos foram detectados? **Especificidade:** $VN/(VN+FP)$ — entre negativos reais, quantos foram corretamente identificados? **F1:** $2VP/(2VP+FP+FN)$ — combina precisão e recall.

**Exemplo:** em detecção de doenças, um falso negativo é uma pessoa doente classificada como saudável; um falso positivo é uma pessoa saudável classificada como doente. Se classes são desbalanceadas, acurácia alta pode esconder muitos erros relevantes. **Precisão** em português às vezes é usada informalmente como acurácia: observe a fórmula exigida.

**Exemplo de código logo abaixo:** substitua VP, VN, FP e FN; a célula calcula todas as métricas. Para matrizes multiclasse, métricas podem ser calculadas por classe e agregadas por média.

[scikit-learn — matriz de confusão](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) · [scikit-learn — métricas](https://scikit-learn.org/stable/modules/model_evaluation.html)

### 3.6.1 Recall e especificidade: o que são positivos e negativos reais?

Primeiro, **defina qual classe será considerada positiva**. Em diagnóstico, podemos chamar de **positivo** o caso em que a doença está presente. Um **positivo real** é alguém que realmente tem a doença; um **negativo real** é alguém que realmente não a tem. A palavra **real** se refere à resposta de referência, não à previsão do modelo.

- **Verdadeiro positivo (VP):** doente classificado como doente.
- **Falso negativo (FN):** doente classificado como saudável; caso perdido.
- **Falso positivo (FP):** saudável classificado como doente; alarme falso.
- **Verdadeiro negativo (VN):** saudável classificado como saudável.

**Recall/sensibilidade = VP / (VP + FN):** de todos os **positivos reais**, quantos o modelo detectou? Se 90 de 100 pessoas doentes foram identificadas, recall = 90%.

**Especificidade = VN / (VN + FP):** de todos os **negativos reais**, quantos foram reconhecidos corretamente? Se 80 de 100 saudáveis foram identificados como saudáveis, especificidade = 80%.

**Precisão (precision) = VP / (VP + FP):** de todos que o modelo **previu como positivos**, quantos realmente eram positivos? O denominador é diferente do recall. **FPR = 1 − especificidade** é a taxa de falsos alarmes; **FNR = 1 − recall** é a taxa de casos perdidos.

**Acurácia pode enganar:** se 99% dos exemplos são negativos, prever sempre negativo gera 99% de acurácia, mas recall = 0% para positivos. **Curva ROC** compara recall e taxa de falsos positivos em diferentes limiares; **curva precision–recall** costuma ser útil para positivos raros.

**Multiclasse:** é possível calcular métricas por classe (*one-vs-rest*) e depois obter médias **macro** (cada classe pesa igualmente), **micro** (soma de contagens) ou **ponderada** (considera frequências). Para segmentação, IoU e Dice complementam métricas de classificação.

[scikit-learn — métricas de classificação](https://scikit-learn.org/stable/modules/model_evaluation.html)

### 3.7 Python na visão computacional

**Python** oferece estruturas de dados, funções e bibliotecas. **NumPy** representa matrizes e realiza cálculos vetorizados; **Matplotlib** visualiza imagens/gráficos; **OpenCV** (`cv2`) lê e transforma imagens e vídeos; **TensorFlow/Keras** e **PyTorch** criam e treinam redes neurais.

Operações típicas do OpenCV: `cv2.imread` lê imagem; `cv2.cvtColor` converte espaço de cores; `cv2.resize` altera dimensões; `cv2.GaussianBlur` suaviza; `cv2.Canny` detecta bordas. **Não confundir detecção de bordas com detecção de objetos:** bordas identificam mudanças locais de intensidade, objetos exigem interpretação adicional.

**Exemplo conceitual:** carregar uma imagem → converter cores → redimensionar → normalizar pixels → alimentar modelo → obter logits/probabilidades → comparar rótulo previsto ao real.

**Cuidado com dependências:** o código de métricas abaixo usa apenas recursos já importados na preparação. Exemplos com OpenCV/TensorFlow exigiriam instalação e suas importações deveriam ser adicionadas **exclusivamente ao bloco inicial**, não às células posteriores.

[OpenCV — documentação](https://docs.opencv.org/4.x/) · [TensorFlow — tutoriais](https://www.tensorflow.org/tutorials)

### Como utilizar o cálculo abaixo

O código seguinte recebe VP (verdadeiros positivos), VN (verdadeiros negativos), FP (falsos positivos) e FN (falsos negativos). Substitua esses quatro valores e execute: imprime acurácia (acertos totais), precisão (acertos entre previstos positivos), recall (positivos reais identificados), especificidade e F1.

In [ ]:
VP, VN, FP, FN = 40, 50, 10, 5  # ALTERE

def dividir(a,b): return a/b if b else float('nan')
acuracia = dividir(VP+VN, VP+VN+FP+FN)
precisao = dividir(VP, VP+FP)
recall = dividir(VP, VP+FN)
especificidade = dividir(VN, VN+FP)
f1 = dividir(2*VP, 2*VP+FP+FN)
for nome, valor in [('Acurácia',acuracia),('Precisão',precisao),('Recall',recall),('Especificidade',especificidade),('F1',f1)]:
    print(f'{nome}: {valor:.4f} ({valor:.2%})')

## 4. Machine learning aplicado ao processamento de linguagem natural (PLN)

### 4.1 O problema do PLN

**PLN** estuda métodos para computadores analisar, compreender e produzir linguagem humana. Exemplos: classificação de textos, análise de sentimentos (positivo/negativo/neutro), tradução, resumo, respostas a perguntas, reconhecimento de entidades (nomes, datas, locais), chatbots e reconhecimento de fala.

**Texto é sequencial:** a ordem das palavras e o contexto influenciam significado. Antes de treinar modelos, palavras e frases são convertidas em **números**: tokens, índices, vetores de frequência, TF-IDF ou *embeddings* (representações vetoriais aprendidas). **Corpus** é uma coleção de textos usada em análise ou treinamento.

**Tokenização** divide texto em unidades; **lematização** obtém forma de dicionário; **stemming** corta partes das palavras sem necessariamente gerar formas válidas; **stopwords** são termos muito frequentes que podem ser removidos em tarefas apropriadas. **NER** (*named entity recognition*) identifica entidades nomeadas; **POS tagging** classifica funções gramaticais.

### 4.2 Redes neurais para texto

**RNN:** recebe tokens em sequência e mantém um **estado oculto**, que representa memória das entradas anteriores. **Pesos são compartilhados ao longo do tempo**. Seu treinamento pode usar **BPTT** (*backpropagation through time*), que propaga gradientes pelos passos da sequência. Vantagem: ordem temporal explícita; limitação: dependências longas e processamento sequencial. **LSTM/GRU** são variantes que melhoram a retenção de contexto.

**CNN para PLN:** aplica filtros a **janelas locais de tokens/embeddings**, capturando padrões como expressões e combinações próximas; pode ser usada em classificação de texto e sentimento. Ao contrário da imagem 2D, normalmente a convolução percorre a dimensão da sequência. Não é exclusiva de visão computacional.

**Transformer:** arquitetura baseada em **atenção**; permite que cada token considere outros tokens da sequência. Ao contrário de RNNs tradicionais, não exige recorrência passo a passo para codificar sequências, favorecendo **paralelização do treinamento**. Usa mecanismos como autoatenção, representações posicionais, conexões residuais e normalização. Nem todo Transformer é encoder-decoder: há modelos encoder-only e decoder-only.

[Hugging Face — conceitos básicos de Transformers](https://huggingface.co/learn/llm-course/pt/chapter1/4) · [Artigo — Attention Is All You Need](https://arxiv.org/abs/1706.03762)

### 4.2.1 Representação numérica de textos: Bag of Words, TF-IDF e embeddings

Um computador não interpreta palavras diretamente: modelos precisam de **representações numéricas**. **Vocabulário** é o conjunto de tokens conhecidos; **tokenização** define o que conta como token (palavras, caracteres ou subpalavras). **Corpus** é a coleção de documentos. Uma palavra pode ter mais de um sentido dependendo do contexto.

**Bag of Words (BoW):** representa um documento pela contagem de termos; geralmente **ignora a ordem**. Exemplo: 'gato caça rato' e 'rato caça gato' produzem contagens iguais. **TF-IDF:** aumenta a importância de palavras frequentes em um documento, mas relativamente raras no corpus. Útil em busca e classificação clássica, mas ainda não compreende contexto profundo.

**One-hot:** vetor com uma posição ativa por palavra; não representa similaridade semântica. **Word2Vec:** aprende vetores densos a partir de contextos; palavras usadas de modo parecido tendem a ter vetores próximos. **Embeddings contextuais** (como os de BERT) podem variar conforme a frase: 'banco' financeiro versus banco de praça.

**Tokenização por subpalavras** ajuda a lidar com palavras raras e vocabulários finitos; uma palavra pode corresponder a vários tokens. **Padding** completa sequências curtas para processamento em lote; **attention mask** indica posições válidas; **truncation** corta sequências longas. Não confunda token, palavra e caractere.

**O próximo código do Tema 4** oferece uma contagem básica de tokens com NumPy/Counter. Para TF-IDF real, consulte scikit-learn; a vetorização deve ser ajustada somente no treino para evitar vazamento.

[scikit-learn — extração de características de texto](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction) · [Hugging Face — tokenizadores](https://huggingface.co/docs/transformers/main_classes/tokenizer)

### 4.2.2 Atenção e Transformers: como a informação circula

**Atenção** permite ponderar a relevância de diferentes tokens para processar um token. Exemplo: em 'Maria pegou o livro porque ela precisava estudar', o termo 'ela' pode se relacionar a 'Maria'. O mecanismo não é uma regra gramatical infalível; aprende padrões estatísticos.

**Self-attention:** tokens da mesma sequência participam de consultas (**queries**), chaves (**keys**) e valores (**values**). Escores de similaridade entre queries e keys passam por softmax e ponderam os values. A expressão clássica é $Attention(Q,K,V)=softmax(QK^T/\sqrt{d_k})V$. **Multi-head attention** aprende vários padrões de relação em paralelo.

**Positional encoding/embedding:** fornece informação sobre ordem, porque a atenção isolada não incorpora automaticamente a posição sequencial. **Feed-forward network:** camadas densas aplicadas às representações dos tokens; **residual connections** ajudam fluxo de informação e gradientes; **layer normalization** estabiliza representações.

**Encoder** produz representações contextualizadas de entrada; **decoder autoregressivo** prevê próximo token sem consultar tokens futuros da saída (*causal mask*); **encoder-decoder** combina leitura de entrada com geração de saída. BERT é exemplo clássico encoder-only; GPT é decoder-only; T5 é encoder-decoder. **Transformer não é sinônimo de GPT**.

**Comparação com RNN:** RNN processa estados ao longo do tempo; Transformer pode processar muitos tokens em paralelo no treinamento, mas a geração autoregressiva continua token a token. Atenção convencional tem custo que cresce aproximadamente com o quadrado do comprimento da sequência, motivando variantes eficientes.

[Artigo Attention Is All You Need](https://arxiv.org/abs/1706.03762) · [Hugging Face — arquitetura de Transformers](https://huggingface.co/learn/llm-course/pt/chapter1/4)

### 4.3 BERT, GPT, pré-treinamento e fine-tuning

**BERT** (*Bidirectional Encoder Representations from Transformers*) é baseado em **encoders** e usa contexto bidirecional. Aplicações clássicas: classificação, reconhecimento de entidades, análise de sentimentos e extração de respostas. **GPT** (*Generative Pre-trained Transformer*) usa arquitetura predominantemente **decoder autoregressiva**: prevê o próximo token com base nos anteriores e gera texto progressivamente. Aplicações: geração, completar texto, diálogo, resumo, tradução e perguntas e respostas.

**Pré-treinamento:** modelo aprende padrões gerais em grande corpus. Em muitos modelos de linguagem isso é **aprendizado autossupervisionado**, pois alvos são derivados dos próprios textos (frequentemente chamado de 'não supervisionado' em materiais introdutórios). **Fine-tuning/ajuste fino:** treinamento adicional para uma tarefa ou domínio específico, modificando alguns ou todos os parâmetros. Nem toda utilização de modelo pré-treinado exige fine-tuning: também é possível usar modelo congelado ou técnicas de prompting.

**Atenção às distinções:** BERT é tradicionalmente mais associado à compreensão bidirecional; GPT à geração autoregressiva; RNN à recorrência e memória de estados; Transformer à atenção e paralelização. **Modelos** não são o mesmo que **frameworks**.

[Artigo BERT](https://arxiv.org/abs/1810.04805) · [Hugging Face — modelos pré-treinados](https://huggingface.co/learn/llm-course/pt/chapter1/3)

### 4.3.1 Pré-treinamento, adaptação e avaliação de modelos de linguagem

**Pré-treinamento autossupervisionado:** o próprio texto fornece alvos de aprendizado. **Masked language modeling** (associado ao BERT original) aprende a prever tokens mascarados usando contexto dos dois lados; **causal language modeling** (associado ao GPT) aprende a prever o próximo token com base nos anteriores.

**Fine-tuning supervisionado:** atualiza pesos com exemplos da tarefa-alvo, como textos rotulados por sentimento. **Congelar backbone** usa representações pré-treinadas sem alterar esses pesos. **PEFT/LoRA** adapta o modelo treinando um subconjunto pequeno de parâmetros. **Prompting/few-shot** instrui um modelo sem necessariamente modificar pesos. **RAG** recupera documentos externos e os fornece como contexto; não é o mesmo que fine-tuning.

**Tarefas e métricas:** classificação de sentimento → precisão/recall/F1; NER → F1 por entidades; geração → avaliação humana e métricas específicas; modelos probabilísticos → perplexidade, com cuidados de tokenização. **Perplexidade não mede diretamente factualidade ou qualidade da resposta.**

**Erros comuns:** alucinação (geração de informação incorreta), viés dos dados, problemas com contexto longo, sensibilidade a idioma/domínio, dados pessoais e vazamento entre treino e avaliação. A confiança expressa em linguagem natural não é uma probabilidade calibrada.

**Escolha de ferramenta:** modelo pré-treinado é útil para começar; fine-tuning requer dados representativos; RAG ajuda quando a resposta depende de documentos atualizados; abordagens tradicionais podem ser melhores em bases pequenas e tarefas simples.

[Hugging Face — treinamento](https://huggingface.co/docs/transformers/training) · [PEFT — documentação](https://huggingface.co/docs/peft/index)

### 4.4 Framework, biblioteca, modelo e API

**Biblioteca:** conjunto de funções/componentes usados pelo programa. **Framework:** estrutura mais abrangente que organiza o desenvolvimento e oferece componentes reutilizáveis, abstração e convenções. Na prática, esses termos às vezes se sobrepõem: o material da disciplina chama ferramentas de PLN de *frameworks*. **Modelo** é o algoritmo/estrutura treinada (por exemplo BERT); **API** é uma interface para acessar funcionalidades.

| Ferramenta | Principal finalidade | Exemplos de uso | O que NÃO é |
|---|---|---|---|
| **spaCy** | pipelines linguísticos eficientes | tokenização, POS, lematização, NER | não é uma arquitetura neural específica |
| **Gensim** | modelagem de tópicos e representações vetoriais | Word2Vec, Doc2Vec, LDA | não é uma biblioteca de interface gráfica |
| **scikit-learn** | ML tradicional e avaliação | TF-IDF, Naive Bayes, regressão logística, métricas, divisão treino/teste | não é exclusivamente para PLN |
| **Transformers (Hugging Face)** | carregar e aplicar modelos pré-treinados | BERT, GPT, tokenizadores, pipelines de tarefas | não é sinônimo da arquitetura Transformer, embora a implemente |
| **TensorFlow/Keras** | construir e treinar redes neurais | modelos de classificação de texto, treinamento e inferência | não é biblioteca específica de tokenização |
| **PyTorch** | treinamento de redes neurais | tensores, autograd, redes personalizadas | não é um modelo de linguagem |

**Componentes reutilizáveis** são vantagem geral dos frameworks; não se restringem a ML. Frameworks aceleram desenvolvimento, mas também podem impor convenções e restrições.

[spaCy — guia](https://spacy.io/usage/spacy-101) · [Gensim — documentação](https://radimrehurek.com/gensim/) · [scikit-learn — classificação de textos](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html) · [Transformers — documentação](https://huggingface.co/docs/transformers/index)

### 4.4.1 Tarefas linguísticas: diferenças que aparecem em avaliações

**Tokenização:** divide texto em unidades. **Segmentação de sentenças:** encontra limites de frases. **Lematização:** obtém forma de dicionário ('correndo' → 'correr'); **stemming:** reduz terminações por regras, podendo gerar formas não existentes. **POS tagging:** atribui classe gramatical contextual ('casa' pode ser substantivo ou forma verbal em contextos diferentes). **NER:** identifica entidades nomeadas (pessoas, organizações, locais). **Dependency parsing:** estima relações sintáticas entre palavras. **Coreferência:** procura expressões que se referem à mesma entidade.

**Análise de sentimentos:** classifica polaridade/opinião; **classificação de intenção:** identifica o objetivo de um pedido; **extração de entidades:** localiza informações estruturadas; **modelagem de tópicos:** identifica temas recorrentes em uma coleção; **similaridade semântica:** compara significados aproximados entre textos; **tradução e resumo:** geram sequências de saída.

**Remoção de stopwords é opcional:** pode ajudar em modelos de frequência, mas prejudicar análise de negação, perguntas e Transformers. **Lematização não é tradução**, **NER não é análise de sentimentos**, **Word2Vec não é um chatbot**. O pipeline deve ser escolhido a partir da tarefa e do tipo de dado.

**Modelos por regras versus estatísticos:** regras são interpretáveis e úteis em domínios restritos; modelos aprendidos podem generalizar, mas dependem de dados, validação e controle de vieses. Combinações híbridas são possíveis.

[spaCy — funcionalidades](https://spacy.io/usage/linguistic-features)

### 4.5 Fluxos práticos de PLN em Python

**Análise linguística com spaCy:** carregar modelo de idioma → processar texto → percorrer tokens → consultar texto, classe gramatical, lema e entidades. Modelos de português precisam estar instalados separadamente; o pacote sozinho não garante um pipeline linguístico completo.

**Tópicos e vetores com Gensim:** preparar frases tokenizadas → treinar Word2Vec para representar palavras em vetores → consultar palavras semelhantes; alternativamente usar LDA para descobrir temas em uma coleção de documentos. Word2Vec aprende representações, **não é um classificador de sentimentos pronto**.

**Classificação tradicional com scikit-learn:** documentos rotulados → separar treino/teste → converter textos com CountVectorizer ou TfidfVectorizer → treinar classificador → prever → avaliar precisão/recall/F1. Ajustar vetorizador apenas no treino evita vazamento de informação.

**Modelos Transformers pré-treinados:** escolher modelo adequado e tokenizador → transformar texto em tokens → inferência com pipeline (por exemplo análise de sentimentos) → interpretar rótulos e escores. Os resultados dependem de idioma, domínio, dados e modelo; **nem todo modelo de sentimentos em inglês funciona bem em português**.

**Exemplo prático sem bibliotecas adicionais:** a célula abaixo ilustra tokenização simples, contagem de palavras e frequência. Ela **não substitui tokenizadores linguísticos** (não trata pontuação/subpalavras corretamente), mas mostra por que texto precisa de representação numérica.

[Hugging Face — pipelines](https://huggingface.co/docs/transformers/main_classes/pipelines) · [spaCy — modelos de idiomas](https://spacy.io/models/pt)

In [ ]:
texto = 'Python ajuda a analisar texto e Python ajuda a aprender'
tokens = texto.lower().split()  # ALTERE o texto acima
frequencias = Counter(tokens)
vocabulario = {palavra: i for i, palavra in enumerate(sorted(frequencias))}
print('Tokens:', tokens)
print('Frequências:', frequencias)
print('Índices do vocabulário:', vocabulario)

### 4.6 Exemplos de aplicações e seleção de abordagens

| Necessidade | Abordagem inicial possível | Cuidados |
|---|---|---|
| Classificar e-mails como spam | TF-IDF + regressão logística | desbalanceamento e mudança de vocabulário |
| Extrair nomes e datas | spaCy/NER | ambiguidade, idioma e domínio |
| Agrupar textos por assunto | embeddings + clustering ou LDA | avaliação qualitativa dos tópicos |
| Buscar documentos relevantes | TF-IDF/BM25 ou embeddings | cobertura e qualidade da recuperação |
| Responder com base em documentos | recuperação + modelo gerador (RAG) | conferir fontes e possíveis erros |
| Traduzir textos | Transformer encoder-decoder | idiomas e terminologia especializada |
| Gerar texto livre | modelo autoregressivo | factualidade, privacidade e custo |
| Processar fala gravada | ASR, depois PLN | ruído, sotaques e erros da transcrição |

**Princípio de projeto:** escolha o menor sistema que resolva o problema com qualidade verificável. A popularidade de uma arquitetura não substitui métricas, testes e análise de erros. Dados e critérios de avaliação são tão importantes quanto a escolha do modelo.

## 5. IA aplicada ao reconhecimento de voz — fundamentos e aplicações

**Objetivo:** entender como o som se transforma em texto, quais componentes tomam cada decisão, como interpretar probabilidades e erros, e como reconhecer as técnicas adequadas a cada problema. Esta seção segue os quatro eixos do PDF da disciplina: **sistemas de reconhecimento de voz; arquitetura; quantização de sinais; implementação em Python**, com explicações complementares e distinções técnicas.

### 5.1 Reconhecimento de fala, voz e locutor: não são sinônimos

- **Reconhecimento automático de fala (ASR, speech-to-text):** descobre **o que foi dito**, transformando áudio em texto. Ex.: ditar 'acender a luz' e obter essas palavras.
- **Reconhecimento de locutor:** descobre **quem falou** (identificação) ou verifica uma identidade declarada (verificação). Não precisa transcrever o conteúdo.
- **Diarização:** determina **quem falou e quando**, dividindo gravação em segmentos atribuídos a locutores (por exemplo, 'Locutor A: 00:00–00:12'). Não exige saber o nome real de cada pessoa.
- **Síntese de fala (TTS, text-to-speech):** caminho inverso: texto → áudio. **Detecção de atividade de voz (VAD):** detecta onde há fala, não o que foi dito. **Identificação de idioma:** detecta a língua falada.

**Categorias de entrada:** palavras isoladas (com pausas), palavras conectadas (sequências curtas), fala contínua (frases corridas) e fala espontânea (hesitações, autocorreções, interrupções). A classificação descreve a **forma da fala**, não o algoritmo de reconhecimento. Aplicações: acessibilidade, ditado, assistentes, inspeções, legendagem, reuniões e automação.

**Desafios reais:** ruído, reverberação, distância do microfone, sotaques, velocidade, pronúncia, termos especializados e privacidade. Melhorar supressão de ruído ajuda na captação; treinar com variedade de sotaques ajuda na generalização. São intervenções complementares, não equivalentes.

[SpeechRecognition — documentação e exemplos](https://pypi.org/project/SpeechRecognition/) · [Google — introdução a fala e áudio](https://cloud.google.com/speech-to-text/docs)

### 5.2 Visão geral: do microfone à palavra reconhecida

**Pipeline clássico de reconhecimento:**

1. **Captação:** microfone converte vibrações do ar em sinal elétrico analógico.
2. **Digitalização:** amostragem (instantes discretos), quantização (amplitudes discretas) e codificação (bits).
3. **Pré-processamento do áudio:** filtragem, redução de ruído, normalização de amplitude e detecção de trechos com fala; nem toda técnica é necessária em todos os sistemas.
4. **Extração de características:** transformar pequenas janelas de áudio em vetores numéricos, por exemplo espectrogramas ou MFCCs.
5. **Modelagem acústica:** estimar correspondências entre características sonoras e unidades de fala (fonemas, estados acústicos ou tokens, conforme a arquitetura).
6. **Modelagem léxica:** relacionar palavras às suas pronúncias, no caso de decodificadores com léxico explícito.
7. **Modelo de linguagem:** estimar quais sequências de palavras/tokens são prováveis no contexto.
8. **Decodificador/busca:** combinar evidência acústica, léxico e probabilidades linguísticas para selecionar uma sequência.
9. **Saída:** transcrição, comando ou legenda, possivelmente com confiança e marcação temporal.

**Exemplo:** a gravação contém 'ligar a luz'. O áudio pode ser ambíguo em determinados sons; o modelo acústico indica quais sons são plausíveis, o léxico associa sons a palavras, o modelo de linguagem avalia sequências e o decodificador escolhe a transcrição mais provável. **A saída do modelo não é garantia de verdade:** erros devem ser medidos com dados de referência.

**Distinção importante:** essa divisão é uma **arquitetura didática clássica**, presente no PDF. Sistemas neurais modernos *end-to-end* podem aprender etapas conjuntamente, dispensar um léxico explícito e usar tokens de subpalavras ou caracteres. Portanto, 'todo sistema necessariamente usa três módulos separados' é uma generalização incorreta.

### 5.3 Modelagem acústica × léxica × linguagem × decodificador

| Componente | Pergunta principal | Entrada típica | Saída típica |
|---|---|---|---|
| **Acústico** | 'Que sons/unidades de fala são compatíveis com este áudio?' | vetores de áudio | escores de unidades acústicas |
| **Léxico/pronúncia** | 'Como esta palavra é pronunciada?' | palavra ou sequência de fonemas | mapeamento palavra ↔ pronúncia |
| **Linguagem** | 'Que sequência de palavras faz sentido/provavelmente ocorre?' | palavras anteriores | probabilidades de próximas palavras |
| **Decodificador** | 'Qual sequência completa é a mais provável?' | evidências e escores dos modelos | texto reconhecido |

**Exemplo de linguagem:** depois de 'quero beber', 'água' pode ser mais provável que 'janela'. Isso não significa que o modelo de linguagem *ouça* o áudio: ele modela regularidades textuais. **Exemplo de léxico:** a palavra escrita 'casa' é associada a uma pronúncia; léxico de ASR é frequentemente um **dicionário de pronúncias**, enquanto morfologia e semântica são outras camadas de análise linguística.

**Observação sobre o PDF:** o material usa 'modelagem léxica' também para significados e morfemas. Para estudar a disciplina, conheça **ambas as acepções**: (a) análise lexical/morfológica de palavras; (b) em ASR clássico, correspondência entre palavras e fonemas. Elas se relacionam, mas não são idênticas.

[CMUSphinx — modelos e dicionários](https://cmusphinx.github.io/wiki/tutorialconcepts/) · [Jurafsky & Martin — Speech and Language Processing](https://web.stanford.edu/~jurafsky/slp3/)

### 5.4 Representação da fala: grafemas, fonemas e morfemas

**Grafema:** unidade da escrita (em exemplos simples, letra). **Fonema:** unidade sonora capaz de distinguir palavras; não é necessariamente igual a uma letra. **Morfema:** menor unidade com significado ou função gramatical. Não confunda **som**, **escrita** e **formação de palavras**.

Exemplos didáticos do PDF: 'canto' tem cinco letras e quatro fonemas em uma análise comum, porque 'an' representa nasalização; 'táxi' tem quatro letras e pode ser analisado com cinco fonemas, pois 'x' representa /ks/. A contagem depende da pronúncia e da convenção fonológica usada.

**Estrutura morfológica:** radical (núcleo de significado: 'feliz' em 'infeliz'), prefixo ('in-'), sufixo ('-mente' em 'felizmente'), desinências (flexões como número e tempo) e vogal temática (elemento que pode ligar radical e desinência). **POS tagging** classifica palavras por função gramatical (substantivo, verbo etc.). Morfologia não é o mesmo que reconhecimento acústico.

**Pré-processamento de texto após transcrição:** tokenização separa unidades; remoção de *stopwords* elimina palavras muito frequentes **quando apropriado**; lematização obtém forma canônica ('falando' → 'falar'); stemming reduz palavras por regras e pode gerar formas não existentes. **Atenção:** remover 'não' pode inverter o sentido de uma frase; nem todo pipeline deve remover stopwords.

**Regras vs aprendizado de máquina:** sistemas baseados em regras exigem conhecimento linguístico codificado; sistemas aprendidos inferem padrões a partir de dados rotulados ou outros sinais de treinamento. Modelos híbridos também existem. A qualidade dos dados e a diversidade de sotaques importam muito.

### 5.5 Amostragem, quantização e codificação — três operações diferentes

O som no ar varia continuamente no tempo e em amplitude. Para armazená-lo digitalmente:

**(1) Amostragem:** mede a amplitude em instantes separados. A **taxa de amostragem** $f_s$ é medida em hertz: $16.000\,\mathrm{Hz}$ significa **16 mil amostras por segundo**. Em 2 segundos, são 32 mil amostras (por canal). A amostragem discretiza o **tempo**; sozinha, não limita a amplitude a níveis inteiros.

**(2) Quantização:** aproxima cada amplitude amostrada para um nível disponível. Com $b$ bits por amostra, um quantizador uniforme convencional dispõe de $L=2^b$ níveis. Assim, **3 bits → 8 níveis; 8 bits → 256 níveis; 16 bits → 65.536 níveis**. Mais níveis geralmente reduzem erro de quantização para a mesma faixa dinâmica, mas aumentam armazenamento; saturação (*clipping*) ocorre quando o sinal excede a faixa representável.

**(3) Codificação:** representa os níveis quantizados em bits e define como serão armazenados/transmitidos (por exemplo, PCM). **CAD/ADC** é o conversor analógico-digital que implementa essas etapas.

**Por que não basta aumentar a taxa de amostragem?** Ela controla a resolução temporal e o limite de frequências representáveis; **bits** controlam a resolução de amplitude. Pelo critério de Nyquist, para representar componentes até $f_{max}$ sem aliasing, precisamos $f_s>2f_{max}$, com filtragem antialiasing adequada. Amostragem e quantização geram erros de naturezas diferentes.

**Quantizador sem memória:** cada amostra é quantizada individualmente, sem depender de amostras anteriores. Nem todos os esquemas avançados de codificação são sem memória.

[NumPy — processamento de arrays](https://numpy.org/doc/stable/) · [SciPy — processamento de sinais](https://docs.scipy.org/doc/scipy/reference/signal.html)

### 5.6 Erro de quantização: definição, cálculo e interpretação

Se $x_i$ é o valor original e $q_i$ o quantizado, o **erro assinado** pode ser definido como $e_i=x_i-q_i$ (algumas fontes usam a convenção oposta). **Erro absoluto:** $|e_i|$; **erro absoluto total:** $\sum_i|e_i|$; **erro quadrático total (SSE):** $\sum_i e_i^2$; **erro quadrático médio (MSE):** $\frac{1}{N}\sum_i e_i^2$. **RMSE:** raiz do MSE.

**Exemplo:** original $[1{,}2;2{,}7;3{,}4]$, quantizado $[1;3;3]$. Erros assinados: $[0{,}2;-0{,}3;0{,}4]$. A soma assinada é $0{,}3$, mas os erros podem se cancelar; por isso, o **erro absoluto total** é $0{,}9$ e o **SSE** é $0{,}29$. **Não confundir** soma assinada de erros com uma medida de distorção.

**Como identificar o cálculo em um exercício:** se o enunciado diz 'diferença entre amostra e quantização', calcule $x-q$; se diz 'erro total' sem especificar, verifique se o material espera soma assinada, soma absoluta ou soma de quadrados. Na prática, SSE/MSE são mais úteis para medir distorção.

**O próximo código** recebe valores originais e aproximados e imprime cada uma dessas métricas. **Altere os dois vetores**, mantendo o mesmo tamanho.

In [ ]:
original = np.array([1.2, 2.7, 3.4])  # ALTERE: valores medidos
quantizado = np.array([1.0, 3.0, 3.0])  # ALTERE: valores quantizados
assert original.shape == quantizado.shape, "Os vetores precisam ter o mesmo tamanho"
erro = original - quantizado
print("Erros assinados:", erro)
print("Soma assinada:", np.sum(erro))
print("Erro absoluto total:", np.sum(np.abs(erro)))
print("SSE:", np.sum(erro ** 2))
print("MSE:", np.mean(erro ** 2))
print("RMSE:", np.sqrt(np.mean(erro ** 2)))
bits = 3  # ALTERE: resolução em bits
print("Níveis disponíveis:", 2 ** bits)

### 5.7 Quantização vetorial: diferença para quantização escalar

**Quantização escalar:** cada amostra é aproximada individualmente para um nível, por exemplo $6{,}3\rightarrow6$. **Quantização vetorial:** trata um **vetor de características** inteiro como uma unidade e o substitui por um **vetor representativo** (*centroide*) escolhido de um **codebook** (dicionário de vetores).

**Exemplo:** um vetor acústico $x=[1{,}1,2{,}9]$ é comparado com centroides $c_1=[1,3]$ e $c_2=[4,5]$. O centroide mais próximo é $c_1$ segundo a distância euclidiana. O índice do centroide pode ser armazenado no lugar do vetor completo, comprimindo dados à custa de erro.

**Fórmula:** $k^*=\operatorname{argmin}_k\|x-c_k\|_2$. Leia: 'encontre o índice $k$ do centroide cuja distância euclidiana ao vetor $x$ é a menor'. **Argmin retorna o índice**, não a distância mínima. Centroides podem ser aprendidos por agrupamento como k-means.

**O próximo código** compara um vetor com centroides; altere $x$ e as linhas de $centroides$.

In [ ]:
x = np.array([1.1, 2.9])  # ALTERE: vetor a representar
centroides = np.array([[1.0, 3.0], [4.0, 5.0], [0.0, 0.0]])  # ALTERE: codebook
assert centroides.shape[1] == x.size, "Dimensões incompatíveis"
distancias = np.linalg.norm(centroides - x, axis=1)
k = int(np.argmin(distancias))
print("Distâncias:", distancias)
print("Índice do centroide mais próximo:", k)
print("Vetor quantizado:", centroides[k])
print("Erro de representação:", np.linalg.norm(x - centroides[k]))

### 5.8 Modelos de Markov e modelos ocultos de Markov (HMM)

**Cadeia de Markov:** modelo de estados no qual a probabilidade do próximo estado depende do **estado atual** (propriedade de Markov de primeira ordem). Uma **matriz de transição** $P$ guarda $P_{ij}=P(S_{t+1}=j\mid S_t=i)$. Em uma matriz organizada por **linhas = origem** e **colunas = destino**, cada linha deve somar 1.

**Exemplo:** estados 'silêncio' e 'fala'. A matriz $[[0{,}8,0{,}2],[0{,}3,0{,}7]]$ significa que, estando em silêncio, a chance de continuar em silêncio é 80% e a de passar a fala é 20%; estando em fala, a chance de passar a silêncio é 30%. A probabilidade após dois passos é calculada por $P^2$, e uma distribuição inicial $\pi$ evolui para $\pi P$.

**Modelo oculto de Markov (HMM):** os **estados internos não são observados diretamente**; observamos emissões/sinais que dependem desses estados. No reconhecimento clássico de fala, estados acústicos podem ser ocultos e características de áudio são observações. Um HMM especifica probabilidades **de transição** e **de emissão**, além da distribuição inicial. **Não confundir:** cadeia de Markov simples ≠ HMM; probabilidades de transição ≠ probabilidades de observação.

**Algoritmos associados:** Viterbi encontra a sequência de estados ocultos mais provável; *forward* calcula probabilidade total das observações; Baum–Welch é um método de treinamento de HMM. A modelagem de palavras como estados, apresentada didaticamente no PDF, é uma simplificação possível, não a única implementação.

**O próximo código** lê a matriz linha-origem/coluna-destino, mostra a próxima distribuição e a matriz de dois passos. Altere os estados e probabilidades.

In [ ]:
estados = ["silêncio", "fala"]  # ALTERE: nomes dos estados
P = np.array([[0.8, 0.2], [0.3, 0.7]], dtype=float)  # ALTERE: linhas=origem, colunas=destino
pi = np.array([1.0, 0.0])  # ALTERE: distribuição atual
assert P.shape == (len(estados), len(estados))
assert np.allclose(P.sum(axis=1), 1) and np.all(P >= 0), "Linhas de P devem somar 1"
assert np.isclose(pi.sum(), 1) and np.all(pi >= 0), "pi deve ser distribuição válida"
print("Próximo estado:", dict(zip(estados, pi @ P)))
print("Após dois passos:", dict(zip(estados, pi @ np.linalg.matrix_power(P, 2))))
print("Matriz de transição em dois passos:\n", np.linalg.matrix_power(P, 2))

### 5.9 N-gramas: sequências de palavras e previsão

Um **n-grama** é uma sequência **contígua** de $n$ unidades, geralmente palavras ou caracteres. **Unigrama:** uma palavra; **bigrama:** duas consecutivas; **trigrama:** três consecutivas. Em 'eu gosto de dados', os trigramas são ('eu','gosto','de') e ('gosto','de','dados'): **não** são todas as combinações possíveis.

**Quantidade em uma frase de $m$ tokens:** $m-n+1$ n-gramas, quando $1\le n\le m$; caso contrário, zero. **Contagem** não é o mesmo que **probabilidade**. Para estimar $P(\text{dados}\mid\text{de})$ com bigramas, conte quantas vezes 'de dados' aparece e divida pela quantidade de vezes em que 'de' aparece como contexto com uma palavra seguinte.

**Modelo de linguagem:** aplica probabilidades condicionais para prever próximas palavras. Para trigrama, $P(w_t\mid w_{t-2},w_{t-1})$. A **regra da cadeia** exata é $P(w_1,w_2,w_3)=P(w_1)P(w_2\mid w_1)P(w_3\mid w_1,w_2)$; o modelo n-grama **aproxima** contextos longos por janelas curtas.

**Limitações:** n pequeno perde contexto; n grande gera dados esparsos e probabilidades zero para sequências não observadas. **Suavização** (por exemplo Laplace/add-one) reduz problemas de zeros, embora modelos avançados usem métodos melhores. N-gramas estatísticos **não exigem rede neural**.

**O próximo código** conta n-gramas de um pequeno corpus e estima uma probabilidade condicional por frequência. Altere o corpus, $n$ e a sequência consultada.

In [ ]:
frases = ["eu gosto de python", "eu gosto de dados", "eu estudo python"]  # ALTERE
n = 2  # ALTERE: 1=unigramas, 2=bigramas, 3=trigramas
sequencias = [frase.lower().split() for frase in frases]
ngramas = Counter(tuple(tokens[i:i+n]) for tokens in sequencias for i in range(len(tokens)-n+1))
print("Contagem de n-gramas:", ngramas)
consulta = ("gosto", "de")  # ALTERE: tupla com n palavras
assert len(consulta) == n, "Consulta deve ter n palavras"
contexto = consulta[:-1]
denominador = sum(qtd for grama, qtd in ngramas.items() if grama[:-1] == contexto)
probabilidade = ngramas[consulta] / denominador if denominador else None
print("P(última palavra | contexto):", probabilidade)

### 5.10 Perplexidade: quão surpreendente é a sequência?

**Perplexidade (PP)** mede a capacidade de um **modelo probabilístico de linguagem** prever uma sequência de tokens. Para uma sequência de $N$ tokens com probabilidades condicionais $p_1,\ldots,p_N$, $PP=\exp\left(-\frac{1}{N}\sum_{i=1}^{N}\ln p_i\right)=\left(\prod_{i=1}^{N}p_i\right)^{-1/N}$. **Quanto menor, melhor**, desde que se comparem modelos no **mesmo conjunto de avaliação e com tokenização compatível**.

**Exemplo:** se a cada passo o modelo atribui probabilidade $0{,}5$ à palavra correta, a perplexidade é 2. Se atribui $0{,}25$, a perplexidade é 4. Isso não significa 'acurácia de 50%' ou '25%': perplexidade **não é taxa de acerto** e seu inverso **não é, em geral, a probabilidade de acertar a transcrição**. É uma medida de incerteza preditiva.

**Perplexidade ≠ WER:** perplexidade avalia o modelo de linguagem; **WER (Word Error Rate)** mede erros da transcrição em relação a um texto de referência: $WER=(S+D+I)/N$, com substituições, exclusões, inserções e $N$ palavras na referência. Um modelo pode ter perplexidade baixa e ainda cometer erros acústicos. **Probabilidade zero** para um token observado gera perplexidade infinita (sem suavização).

**O próximo código** calcula perplexidade com logaritmos, evitando produtos de números minúsculos. Substitua a lista pelas probabilidades que o modelo atribuiu às palavras corretas.

In [ ]:
probabilidades = [0.5, 0.25, 0.5]  # ALTERE: probabilidades condicionais dos tokens corretos
assert probabilidades and all(0 < p <= 1 for p in probabilidades), "Use probabilidades em (0, 1]"
perplexidade = math.exp(-sum(math.log(p) for p in probabilidades) / len(probabilidades))
print("Perplexidade:", perplexidade)
print("Interpretação: menor é melhor para a mesma tarefa, corpus e tokenização")

### 5.11 Diarização: identificar quem falou e quando

Em uma reunião com duas pessoas, uma transcrição comum pode conter apenas 'Bom dia. Podemos começar?'. **Diarização** acrescenta **intervalos e rótulos de locutores**: '[00:00–00:02] Locutor A: Bom dia' e '[00:03–00:05] Locutor B: Podemos começar?'.

**Etapas usuais:** (1) detectar trechos de fala; (2) localizar mudanças de locutor; (3) extrair representações de voz (*speaker embeddings*); (4) agrupar trechos da mesma pessoa; (5) alinhar rótulos e transcrições. **Segmentação** encontra fronteiras; **agrupamento** reúne segmentos do mesmo falante. **Diarização não identifica automaticamente o nome civil** do locutor: os rótulos A/B são normalmente anônimos.

**Diferenças rápidas:** ASR = 'o que'; diarização = 'quem e quando'; verificação = 'é a pessoa alegada?'; identificação = 'qual pessoa de uma base?'.

[pyannote.audio — diarização](https://github.com/pyannote/pyannote-audio)

### 5.12 Python para reconhecimento de fala: bibliotecas e fluxo

**SpeechRecognition** (pacote instalável `SpeechRecognition`, módulo `speech_recognition`) fornece interface para reconhecer fala usando diferentes mecanismos. **PyAudio** permite capturar som de microfones; pode exigir instalação específica no sistema. **NumPy** manipula sinais numéricos; **SciPy** oferece processamento de sinais; **Matplotlib** plota formas de onda; **Tkinter** cria interfaces gráficas. Nenhuma dessas três últimas bibliotecas, sozinha, é um reconhecedor de fala. **`voice_recognition` não é o nome do módulo usado pelo pacote SpeechRecognition**.

**Fluxo de implementação:** (1) instalar dependências; (2) criar `sr.Recognizer()`; (3) abrir `sr.Microphone()` ou `sr.AudioFile(...)`; (4) captar áudio com `listen` ou `record`; (5) transcrever com mecanismo escolhido, por exemplo `recognize_google`; (6) tratar `UnknownValueError` (fala incompreensível) e `RequestError` (problema no serviço); (7) utilizar o texto.

**Instalação opcional no ambiente Jupyter:** `%pip install SpeechRecognition`. Para microfone, instale também PyAudio conforme seu sistema. O serviço usado por `recognize_google` depende de conectividade e condições de uso; não presuma funcionamento offline nem disponibilidade irrestrita.

**Para respeitar a organização deste notebook**, a importação opcional do pacote fica **somente na célula inicial**; a célula abaixo contém apenas o processamento. Se o pacote não estiver instalado, a célula exibe instruções em vez de falhar na importação.

[SpeechRecognition — PyPI e exemplos](https://pypi.org/project/SpeechRecognition/) · [SpeechRecognition — repositório](https://github.com/Uberi/speech_recognition)

In [ ]:
if sr is None:
    print("Opcional: execute %pip install SpeechRecognition e reinicie o kernel.")
else:
    caminho_wav = ""  # ALTERE: caminho de um arquivo WAV/AIFF/FLAC; vazio não executa
    if not caminho_wav:
        print("Defina caminho_wav para transcrever um arquivo de áudio.")
    else:
        reconhecedor = sr.Recognizer()
        try:
            with sr.AudioFile(caminho_wav) as fonte:
                audio = reconhecedor.record(fonte)
            texto_reconhecido = reconhecedor.recognize_google(audio, language="pt-BR")
            print("Transcrição:", texto_reconhecido)
        except sr.UnknownValueError:
            print("Não foi possível compreender a fala.")
        except sr.RequestError as erro_servico:
            print("Falha no serviço de reconhecimento:", erro_servico)
        except (OSError, ValueError) as erro_arquivo:
            print("Problema no arquivo de áudio:", erro_arquivo)

### 5.13 Como escolher a técnica adequada

| Situação | Técnica/conceito mais relacionado | Por quê |
|---|---|---|
| Converter áudio em texto | ASR / decodificação | reconhece palavras faladas |
| Descobrir quem falou em cada trecho | diarização | segmenta e agrupa por locutor |
| Confirmar identidade por voz | verificação de locutor | compara com identidade alegada |
| Descobrir a língua falada | identificação de idioma | classifica o idioma |
| Gerar áudio a partir de texto | síntese de fala (TTS) | caminho texto → som |
| Reduzir interferência de ruído | filtragem / supressão de ruído | melhora qualidade da entrada |
| Transformar som contínuo em amostras | amostragem | discretiza o tempo |
| Aproximar amplitudes a níveis finitos | quantização | discretiza valores de amplitude |
| Relacionar áudio a unidades de fala | modelo acústico | aprende evidência sonora |
| Relacionar grafia a pronúncia | léxico de pronúncias | mapeia palavra ↔ fonemas |
| Avaliar plausibilidade de frase | modelo de linguagem | estima probabilidades de sequências |
| Escolher sequência final de palavras | decodificador | combina evidências e busca |
| Agrupar palavras consecutivas | n-gramas | contexto de n tokens |
| Estimar próxima palavra/estado | n-grama / cadeia de Markov | probabilidade condicional |
| Avaliar previsão probabilística de texto | perplexidade | incerteza do modelo de linguagem |
| Avaliar erros na transcrição | WER | inserções, exclusões e substituições |
| Representar vetor pelo centroide mais próximo | quantização vetorial | compressão / agrupamento |

### 5.14 Referências e limites do material

**Base da disciplina:** PDF *IA aplicada ao reconhecimento de voz* (Estácio, 44 páginas), especialmente sistemas de voz, HMM, n-gramas, diarização, arquitetura, modelagem acústica/léxica/linguística, perplexidade, quantização escalar/vetorial e implementação Python. **Ampliações explicativas:** distinção ASR/locutor/VAD, modelos end-to-end, diferença entre amostragem e quantização, SSE/MSE, Nyquist, WER e ressalvas sobre perplexidade; esses complementos não são transcrições do PDF.

**Nota de precisão:** algumas simplificações didáticas do PDF são tratadas com ressalvas (por exemplo, a associação entre inverso da perplexidade e taxa de acerto; e o uso amplo de 'léxico'). Para exercícios, identifique o conceito pedido e também considere a formulação adotada pela disciplina.

## Resumo geral para consulta

**Aprendizado:** supervisionado = dados rotulados; não supervisionado = padrões sem rótulos; parâmetros = pesos aprendidos; hiperparâmetros = escolhas do treinamento.

**CNN:** convolução extrai características; pooling reduz resolução; ReLU ativa; loss mede erro; otimizador atualiza pesos. VGGNet = blocos de convoluções 3×3 e pooling, com topo totalmente conectado clássico; GoogLeNet = módulos Inception com ramos paralelos **internos**.

**Visão computacional:** classificação = rótulo; detecção = localização; segmentação semântica = classes por pixel; instâncias = indivíduos por pixel; GAN = gerador + discriminador; RNN = sequências; nuvem = infraestrutura escalável.

**PLN:** RNN = memória sequencial; CNN textual = padrões locais; Transformer = atenção; BERT = contexto bidirecional; GPT = geração autoregressiva. spaCy = análise linguística; Gensim = tópicos/vetores; scikit-learn = ML clássico; Hugging Face Transformers = modelos pré-treinados.

**Voz:** modelo acústico = som; léxico = pronúncia; linguagem = sequências de palavras; n-grama = sequência de n tokens; perplexidade = surpresa preditiva; matriz de estados = probabilidades de transição.

## Índice alfabético de conceitos

Use **Ctrl+F** pelo termo ou consulte a seção indicada. As entradas apontam para explicações no corpo do material.

| Conceito | Seção | Ideia central |
|---|---|---|
| Acurácia, precisão, recall, F1 | 3.6 | métricas da matriz de confusão |
| Aprendizado supervisionado | 1; 2; 3.5 | rótulos orientam aprendizado |
| Backpropagation | 2 | cálculo de gradientes |
| BERT / GPT | 4.3 | compreensão / geração |
| CNN / convolução / pooling | 2; 3.4 | filtros / redução espacial |
| Dropout / overfitting | 1; 2 | regularização / generalização |
| Fine-tuning | 2; 4.3 | adaptar modelo pré-treinado |
| Função de erro / loss | 2 | comparação previsão-alvo |
| GAN | 3.4 | gerador e discriminador |
| GoogLeNet / Inception | 2 | módulos com ramos paralelos internos |
| Learning rate / momentum | 2 | passo e histórico de atualização |
| Matriz de confusão | 3.6 | classificação real versus prevista |
| N-gramas / perplexidade | 5.9–5.10 | sequência e probabilidade |
| OpenCV / NumPy | 3.7 | processamento de imagens e matrizes |
| Pesos sinápticos | 2 | parâmetros aprendidos |
| ReLU / sigmoid / softmax | 2 | funções de ativação |
| RNN / LSTM / BPTT | 4.2 | sequência, memória e treino temporal |
| Segmentação semântica / instâncias | 3.3 | rótulos de pixels e objetos |
| spaCy / Gensim / scikit-learn | 4.4–4.5 | ferramentas de PLN |
| Transformer / atenção | 4.2 | dependências entre tokens |
| Transição de estados / HMM | 5.8 | probabilidade de mudança e estados ocultos |
| VGGNet / fully connected | 2 | blocos 3×3 e classificador |

| Amostragem / quantização / bits | 5.5–5.6 | tempo, níveis de amplitude e erros |
| Quantização vetorial / centroides | 5.7 | representação por vetor próximo |
| Modelo acústico / léxico / linguagem | 5.3 | som, pronúncia e sequência |
| Decodificador | 5.2–5.3 | seleção da transcrição |
| Diarização / ASR / TTS | 5.1; 5.11 | quem falou, o que foi dito e síntese |
| SpeechRecognition / PyAudio | 5.12 | transcrição e captura de áudio |
| WER | 5.10 | erros de transcrição |
| Atenção / self-attention / QKV | 4.2.2 | dependências entre tokens |
| Bag of Words / TF-IDF / embeddings | 4.2.1 | representação numérica de textos |
| Camada Dense / fully connected | 2 | representação intermediária e parâmetros |
| IoU / Dice / mAP / NMS | 3.3.2; 3.6.2 | localização e segmentação |
| LoRA / RAG / prompting | 4.3.1 | formas de adaptar ou consultar modelos |
| Pixel / RGB / BGR / normalização | 3.3.1 | representação e preparação de imagens |
| ROC / especificidade / recall | 3.6.1 | métricas e erros por classe |
| Vazamento de dados / baseline | 1.1–1.2 | avaliação e generalização |

### Materiais de referência

**Base dos Temas 3 e 4:** PDFs da Estácio *IA aplicada à visão computacional* (54 páginas) e *Machine Learning Aplicado para PLN* (46 páginas), disponibilizados com este estudo. As seções preservam a cobertura de fundamentos, arquiteturas, aplicações e uso de Python; explicações complementares são síntese didática, não transcrição.

**Documentação externa:** [Google ML Crash Course](https://developers.google.com/machine-learning/crash-course), [Stanford CS231n](https://cs231n.github.io/convolutional-networks/), [OpenCV](https://docs.opencv.org/4.x/), [spaCy](https://spacy.io/usage/spacy-101), [Hugging Face](https://huggingface.co/learn/llm-course/pt/chapter1/1).

**Versão 0.5:** Temas 1–4 aprofundados: fundamentos de ML, Dense intermediária, otimização, visão computacional, métricas, representação de texto, Transformers e avaliação. O Tema 5 permanece disponível com sua cobertura ampliada.